In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
songs_df = pd.read_csv('..\Data\Music Info.csv', usecols=["track_id","name","artist","spotify_preview_url"])
songs_df.head()

,track_id,name,artist,spotify_preview_url
0,TRIOREW128F424EAF0,Mr. Brightside,The Killers,https://p.scdn.co/mp3-preview/4d26180e6961fd46...
1,TRRIVDJ128F429B0E8,Wonderwall,Oasis,https://p.scdn.co/mp3-preview/d012e536916c927b...
2,TROUVHL128F426C441,Come as You Are,Nirvana,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...
3,TRUEIND128F93038C4,Take Me Out,Franz Ferdinand,https://p.scdn.co/mp3-preview/399c401370438be4...
4,TRLNZBD128F935E4D8,Creep,Radiohead,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...


In [13]:
%pip install dask[dataframe]

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 15.7 MB/s  0:00:00

   ---------------------------------------- 0/4 [toolz]
   ---------------------------------------- 0/4 [toolz]
   ---------- ----------------------------- 1/4 [locket]
   -------------------- ------------------- 2/4 [partd]
   -------------------- ------------------- 2/4 [partd]
   ------------------------------ --------- 3/4 [dask]
   ------------------------------ --------- 3/4 [dask]
   ------------------------------ --------- 3/4 [dask]
   ------------------------------ --------- 3/4 [dask]
   ------------------------------ --------- 3/4 [dask]
   ------------------------------ --------- 3/4 [dask]
   ------------------------------ --------- 3/4 [dask]
   ------------------------------ --------- 3/4 [dask]
   ------------------------------ --------- 3/4 [dask]
   ------------------------------ --------- 3/4 [dask]
   ----------------------

In [24]:
import dask.dataframe as dd


df = dd.read_csv('../Data/User Listening History.csv')

In [25]:
df.head()

,track_id,user_id,playcount
0,TRIRLYL128F42539D1,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1
1,TRFUPBA128F934F7E1,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1
2,TRLQPQJ128F42AA94F,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1
3,TRTUCUY128F92E1D24,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1
4,TRHDDQG12903CB53EE,b80344d063b5ccb3212f76538f3d9e43d87dca9e,1


In [26]:
df.npartitions

9

In [27]:
df.visualize(tasks=True)

RuntimeError: No visualization engine detected, please install graphviz or ipycytoscape

In [28]:
unique_tracks = df.loc[:,"track_id"].nunique()

unique_tracks = unique_tracks.compute()

unique_tracks

np.int64(30459)

In [29]:
unique_users = df.loc[:,"user_id"].nunique()

unique_users = unique_users.compute()

unique_users

np.int64(962037)

In [31]:
unique_track_ids = df.loc[:,"track_id"].unique().compute()

unique_track_ids = unique_track_ids.tolist()

In [32]:
filtered_songs = songs_df[songs_df["track_id"].isin(unique_track_ids)]

filtered_songs.reset_index(drop=True, inplace=True)

In [33]:
filtered_songs

,track_id,name,artist,spotify_preview_url
0,TRIOREW128F424EAF0,Mr. Brightside,The Killers,https://p.scdn.co/mp3-preview/4d26180e6961fd46...
1,TRRIVDJ128F429B0E8,Wonderwall,Oasis,https://p.scdn.co/mp3-preview/d012e536916c927b...
2,TRUEIND128F93038C4,Take Me Out,Franz Ferdinand,https://p.scdn.co/mp3-preview/399c401370438be4...
3,TRXOGZT128F424AD74,Karma Police,Radiohead,https://p.scdn.co/mp3-preview/5a09f5390e2862af...
4,TRUJIIV12903CA8848,Clocks,Coldplay,https://p.scdn.co/mp3-preview/24c7fe858b234e3c...
...,...,...,...,...
30454,TRXWSIN128F9339A11,Infinite Love Song,Maximilian Hecker,https://p.scdn.co/mp3-preview/8b3d529025fe3c60...
30455,TRPIGDW12903CDEB2D,Slip of the Lip,Fact,https://p.scdn.co/mp3-preview/cf64490291f9a600...
30456,TRQYCFV128F9322F50,Ryusei Rocket,アンティック-珈琲店-,https://p.scdn.co/mp3-preview/d2668a5a3e0b1fda...
30457,TRHQCSH128F42724B7,Colors Of The Wind,ACIDMAN,https://p.scdn.co/mp3-preview/8e22a7052ef3ecf7...


In [35]:
import dask.dataframe as dd
import numpy as np
from scipy.sparse import csr_matrix

df = dd.read_csv("../Data/User Listening History.csv")

df['playcount'] = df['playcount'].astype(np.float64)
df = df.categorize(columns=['user_id', 'track_id'])

user_mapping = df['user_id'].cat.codes
track_mapping = df['track_id'].cat.codes

df = df.assign(
    user_idx=user_mapping,
    track_idx=track_mapping
)

In [37]:
interaction_array = df.groupby(['track_idx', 'user_idx'])['playcount'].sum().reset_index()

In [38]:
interaction_array = interaction_array.compute()

In [40]:
interaction_array

,track_idx,user_idx,playcount
0,0,15780,3.0
1,0,76968,1.0
2,0,134525,2.0
3,0,231541,1.0
4,0,305348,1.0
...,...,...,...
9711296,30458,902360,1.0
9711297,30458,913310,1.0
9711298,30458,922319,1.0
9711299,30458,925779,1.0


In [39]:
row_indices = interaction_array['track_idx']
col_indices = interaction_array['user_idx']
values = interaction_array['playcount']

In [41]:
n_tracks = unique_tracks
n_users = unique_users

sparse_matrix = csr_matrix((values, (row_indices, col_indices)), shape=(n_tracks, n_users))

print("Sparse matrix shape:", sparse_matrix.shape)
print("Non-zero elements:", sparse_matrix.nnz)

Sparse matrix shape: (30459, 962037)
Non-zero elements: 9711301


In [42]:
from sklearn.metrics.pairwise import cosine_similarity

In [43]:
np.where(df['track_id'].cat.categories == "TROINZB128F932F740")

(array([17018]),)

In [44]:
ind = 17018

In [45]:
input_array = sparse_matrix[ind]

input_array

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 610 stored elements and shape (1, 962037)>

In [47]:
similarity_scores = cosine_similarity(input_array, sparse_matrix)

In [48]:
np.sort(similarity_scores)[-6:][::-1]

array([[0.        , 0.        , 0.        , ..., 0.07217127, 0.08225488,
        1.        ]], shape=(1, 30459))

In [49]:
similarity_scores.shape

(1, 30459)

In [50]:
np.argsort(similarity_scores.ravel())[-6:][::-1]

array([17018, 24529, 28964,  3620,  9470,  7882])

In [51]:
recommendations = df['track_id'].cat.categories[np.argsort(similarity_scores.ravel())[-6:][::-1]]

In [52]:
filtered_songs[filtered_songs["name"] == "Crazy in Love"]

,track_id,name,artist,spotify_preview_url
3337,TROINZB128F932F740,Crazy in Love,Beyoncé,https://p.scdn.co/mp3-preview/807828ea7070bda7...


In [61]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

def collaborative_recommendation(song_name, user_data, songs_data, interaction_matrix, k=5):
    
    # Step 1: fetch song row (safe matching)
    song_row = songs_data[songs_data["name"].str.lower() == song_name.lower()]
    
    if song_row.empty:
        raise ValueError(f"Song '{song_name}' not found in dataset")
    
    # Step 2: track_id (avoid .values.item())
    input_track_id = song_row["track_id"].iloc[0]
    print("Input track_id:", input_track_id)
    
    # Step 3: get index safely from categorical
    try:
        ind = np.where(user_data['track_id'].cat.categories == input_track_id)[0][0]
    except IndexError:
        raise ValueError("track_id not found in categorical index")
    
    print("Index:", ind)
    
    # Step 4: input vector
    input_array = interaction_matrix[ind]
    
    # Step 5: similarity computation
    similarity_scores = cosine_similarity(input_array, interaction_matrix).ravel()
    
    # Step 6: top-k indices (exclude itself)
    top_k_idx = np.argsort(similarity_scores)[::-1][1:k+1]
    
    recommendation_track_ids = user_data['track_id'].cat.categories[top_k_idx]
    
    # Step 7: scores
    top_scores = similarity_scores[top_k_idx]
    
    print("Recommended IDs:", recommendation_track_ids)
    print("Scores:", top_scores)
    
    # Step 8: build result table
    temp_df = pd.DataFrame({
        "track_id": recommendation_track_ids,
        "score": top_scores
    })
    
    # Step 9: merge with songs data
    top_k_songs = (
        songs_data
        .merge(temp_df, on="track_id")
        .sort_values(by="score", ascending=False)
        .drop(columns=["track_id", "score"])
        .reset_index(drop=True)
    )
    
    return top_k_songs

In [63]:
collaborative_recommendation(song_name="Love",
                             user_data=df,
                             songs_data=filtered_songs,
                             interaction_matrix=sparse_matrix)

Input track_id: TROHCFR128F92DDDA9
Index: 16868
Recommended IDs: Index(['TRSZBPP128F42507C0', 'TRCJPGW128F92DDDA2', 'TREYOMU128F149F9B1',
       'TRJSQQT128F149F9B4', 'TRKOYXI128F423381A'],
      dtype='string', name='track_id')
Scores: [0.39584764 0.36009862 0.33725991 0.33435316 0.30087798]


,name,artist,spotify_preview_url
0,I Got This Down,Simian Mobile Disco,https://p.scdn.co/mp3-preview/0a65b94fe0c24a68...
1,Hotdog,Simian Mobile Disco,https://p.scdn.co/mp3-preview/c3743021f7401981...
2,She's Good For Business,MSTRKRFT,https://p.scdn.co/mp3-preview/2ac119415b4a2ae6...
3,Street Justice,MSTRKRFT,https://p.scdn.co/mp3-preview/2bcf6204e56bbda5...
4,Deny Selected,Boys Noize,https://p.scdn.co/mp3-preview/35af1563cda85455...
